# EndekGAN-T2I — Unified Experiment Orchestrator

**The only notebook you need to start the experimental workflow.** It *calls*, rather than duplicates, the archived source notebooks and experiment scripts in their proper order.

### What it reproduces
Fixed Endek data audit (370 images) → category/hash/Transformer internal GANs (six conditions × three seeds) → adapted DF-GAN–MiniLM (three seeds) → image-level and semantic diagnostics → paper evidence validation → RQ1, RQ2, RQ3.

**One integrated study:** older CPU and newer CUDA execution are computational stages, not separate research methods. Historical recovery and prototype notebooks are archived but are **not replayed as extra experiments**.

**Safe default:** recompute the public 21-run numerical evidence and display its figures; no GPU training, model download, checkpoint mutation, or original Endek-image redistribution. Switch individual flags only when the prerequisites and processing cost are acceptable.

## 0. Configure stage switches and research paths

In [ ]:
from pathlib import Path
import os, sys, json, pandas as pd
from IPython.display import display, Markdown, Image as NotebookImage
SEARCH=[Path.cwd().resolve(),Path.cwd().resolve().parent]
REPO=next((p for p in SEARCH if (p/"scripts"/"workflow.py").is_file()),None)
if REPO is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks folder.")
sys.path.insert(0,str(REPO/"scripts"))
from workflow import summarize_paths, stage, SOURCE, DATASET, WORK

# SAFE DEFAULT: no model training is performed.
RUN_DATASET_AUDIT = False
RUN_INTERNAL_18_TRAINING = False       # requires authorized data and explicit training token
RUN_EXTERNAL_3_TRAINING = False       # requires CUDA, pinned third-party code and split bundle
RUN_IMAGE_LEVEL_EVALUATION = False   # evaluate saved outputs (creates private local artifacts)
RUN_MASTER_ARCHIVE_RECAP = False     # figures/tables from existing private archive
RUN_PUBLIC_REANALYSIS = True         # safe, CPU-only, runs from public 21-row CSV

print("SOURCE ROOT:",SOURCE)
print("PRIVATE DATA ROOT:",DATASET)
print("LOCAL WORK:",WORK)
print("TRAINING ACTIVE:", RUN_INTERNAL_18_TRAINING or RUN_EXTERNAL_3_TRAINING)

## 1. Verify the research archive and the publication-safe evidence

In [ ]:
status=summarize_paths()
display(pd.DataFrame([{"check":k,"value":str(v)} for k,v in status.items()]))
if not status["public_summary"]:
    raise FileNotFoundError("The public numerical evidence CSV is missing.")
print("Public aggregated 21-run evidence exists. Training does not run unless explicitly selected.")

## 2. Data curation and structured Endek prompt analysis

In [ ]:
# Calls notebooks/01_dataset_and_prompt_audit.ipynb when explicitly enabled.
# Output is redirected to ignored local_runs/dataset_audit, not into the source dataset.
data_stage=stage("dataset",enabled=RUN_DATASET_AUDIT)
print(data_stage)

## 3. Internal controlled experiments: conditioning × seam × seed

In [ ]:
# Calls notebooks/02_internal_conditioning_seam_ablation.ipynb.
# SIX internal settings × THREE seeds = 18 runs.
# Requires ENDEKGAN_ALLOW_TRAINING=YES_RETRAIN_EXPLICITLY in the kernel environment.
internal_stage=stage("internal",enabled=RUN_INTERNAL_18_TRAINING)
print(internal_stage)

## 4. External adapted DF-GAN–MiniLM comparison

In [ ]:
# Calls notebooks/03_external_dfgan_minilm.ipynb.
# Executes 3 separately trained seeds; StyleGAN2-ADA is NOT trained/claimed.
# The seed-2025 historical recovery is provenance only, not a 4th result.
external_stage=stage("external",enabled=RUN_EXTERNAL_3_TRAINING)
print(external_stage)

## 5. Image-level evaluation of saved generations

In [ ]:
# Uses scripts/evaluate_existing_outputs.py (actual code used by the archived
# evaluation notebook), operating on 128 images ×4 classes ×7 models ×3 seeds.
# Results are written to ignored local_runs/image_evaluation. FID/KID stay separate.
evaluation_stage=stage("image_evaluation",enabled=RUN_IMAGE_LEVEL_EVALUATION)
print(evaluation_stage)

## 6. Reconstruct original experiment figures, tables, and provenance

In [ ]:
# Calls notebooks/05_master_recap_original.ipynb; reads completed archives.
# Writes new derivative tables/figures only under ignored local_runs/master_recap.
recap_stage=stage("recap",enabled=RUN_MASTER_ARCHIVE_RECAP)
print(recap_stage)

## 7. Recompute and verify the full numerical comparison (default)

In [ ]:
# Calls scripts/reanalyze_evidence.py. This is safe on a public GitHub clone.
evidence_stage=stage("published_summary",enabled=RUN_PUBLIC_REANALYSIS)
print(evidence_stage)

## 8. Verify the resulting RQ1–RQ3 answers

In [ ]:
if RUN_PUBLIC_REANALYSIS:
    table=pd.read_csv(WORK/"reanalysis"/"VERIFIED_SEVEN_MODEL_SUMMARY.csv")
    display(table.round(5))
    diag=json.loads((WORK/"reanalysis"/"REANALYSIS_VALIDATION.json").read_text(encoding="utf-8"))
    assert diag["runs"]==21 and diag["models"]==7 and diag["all_crosschecks_passed"]
    print(f"RQ2: seam-gap reduction with class = {diag['seam_reduction_pct_with_class']:.1f}%")
    print(f"RQ2: seam-gap reduction without class = {diag['seam_reduction_pct_without_class']:.1f}%")
    display(Markdown((WORK/"reanalysis"/"RQ1_RQ2_RQ3_EVIDENCE_REPORT.md").read_text(encoding="utf-8")))
else:
    print("Public numerical reanalysis disabled. No claims recomputed in this session.")

## 9. Quantitative figure preview

In [ ]:
if RUN_PUBLIC_REANALYSIS:
    for name in ("FIG_RECALCULATED_FID.png","FIG_RECALCULATED_KID.png","FIG_RECALCULATED_SEAM_GAP.png"):
        p=WORK/"reanalysis"/name
        assert p.is_file()
        display(Markdown("**"+name+"**"))
        display(NotebookImage(filename=str(p),width=850))

## 10. Completion and reproducibility boundaries

The master orchestrator **does not simulate or fabricate missing experiments**. Archived FID/KID results are exploratory because the test set contains 37 real images. Lower pixel seam discontinuity does not establish cultural authenticity, text compliance, or weaving feasibility. CLIP values exist for seed 42 only, no completed blinded expert scoring exists, and StyleGAN2-ADA has not been trained under this protocol.

To run an expensive stage, read docs/EXECUTION_MODES.md, supply the authorized private data and an isolated output directory, set the explicit training-authorization environment variable, and change the relevant switch in Cell 0. Do not describe unexecuted stages as validated.

## 11. Display and cross-check all Figure 1–10 manuscript panels

This step displays **exactly the ten manuscript figure assets** stored in the GitHub repository root under paper_figures. This is a traceability check and does **not** rerun image generation. The image-level source code is provided in scripts/evaluate_existing_outputs.py, and independent full training requires authorized data.

In [ ]:
from pathlib import Path
import json, hashlib
paper_root=REPO.parent
manifest_path=paper_root/"paper_figures"/"FIGURE_MANIFEST.json"
if not manifest_path.is_file():
    print("Paper image gallery not present; the master notebook package can still re-analyze numerical evidence.")
else:
    manuscript_figures=json.loads(manifest_path.read_text(encoding="utf-8"))
    assert [int(m["figure"]) for m in manuscript_figures]==list(range(1,11))
    for fig in manuscript_figures:
        p=paper_root/fig["file"]
        assert p.is_file(),p
        assert hashlib.sha256(p.read_bytes()).hexdigest()==fig["sha256"]
        display(Markdown(f"**Figure {fig['figure']}.** {fig['caption']}"))
        display(NotebookImage(filename=str(p),width=850))
    print("ALL TEN MANUSCRIPT FIGURES VERIFIED:",len(manuscript_figures))
